# Experiment: MLP-5 on CIFAR-10
**Google Colab A100** — Runtime → Change runtime type → A100

**Purpose:** Disentangle architecture vs dataset as drivers of the feature norm threshold.

| Setting | Architecture | Dataset | fn at T_NC |
|---|---|---|---|
| Known | MLP-5 | MNIST | 1.052 |
| **This** | **MLP-5** | **CIFAR-10** | **?** |
| Known | ResNet-20 | CIFAR-10 | 1.515 |

**Protocol:** Phase 1 = CE 200 ep → ≥99% train acc. Phase 2 = MSE up to 600 ep → NC1 < 0.01.

**Est. time: ~3 hrs** (CIFAR-10 is harder; budget extended to 600 ep)

**Outputs:** `mlp5_cifar10_summary.csv`, `mlp5_cifar10_s{0,1,2}.csv`

In [1]:
import torch, torchvision, time
import torchvision.transforms as T
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from google.colab import files

torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
DEVICE = 'cuda'
assert torch.cuda.is_available(), 'No GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
MNIST_MLP5_FN   = 1.052
CIFAR_RESNET_FN = 1.515
print(f'Known thresholds — MNIST MLP-5: {MNIST_MLP5_FN}  CIFAR ResNet-20: {CIFAR_RESNET_FN}')


GPU: NVIDIA A100-SXM4-40GB
Known thresholds — MNIST MLP-5: 1.052  CIFAR ResNet-20: 1.515


In [2]:
mean = (0.4914, 0.4822, 0.4465)
std  = (0.2470, 0.2435, 0.2616)
transform = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
trainset  = torchvision.datasets.CIFAR10('/tmp/data', train=True,
                                          download=True, transform=transform)
testset   = torchvision.datasets.CIFAR10('/tmp/data', train=False,
                                          download=True, transform=transform)
train_loader = DataLoader(trainset, batch_size=512, shuffle=True,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=1024, shuffle=False,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)
print(f'CIFAR-10 loaded: {len(trainset):,} train / {len(testset):,} test')


100%|██████████| 170M/170M [00:10<00:00, 15.9MB/s]


CIFAR-10 loaded: 50,000 train / 10,000 test


In [3]:
class MLP(nn.Module):
    def __init__(self, in_dim=3*32*32, width=512, num_classes=10):
        super().__init__()
        layers = [nn.Flatten(), nn.Linear(in_dim, width), nn.ReLU()]
        for _ in range(4):
            layers += [nn.Linear(width, width), nn.ReLU()]
        self.body   = nn.Sequential(*layers)
        self.head   = nn.Linear(width, num_classes)
        self._feats = None
        self.body.register_forward_hook(
            lambda m, i, o: setattr(self, '_feats', o.detach()))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)
    def forward(self, x): return self.head(self.body(x))
    def get_features(self, x): self(x); return self._feats
    def get_classifier_weights(self): return self.head.weight.detach()

m = MLP()
print(f'MLP-5 width=512 CIFAR-10: {sum(p.numel() for p in m.parameters())/1e6:.2f}M params')
del m


MLP-5 width=512 CIFAR-10: 2.63M params


In [4]:
@torch.no_grad()
def compute_nc(model, loader, K=10):
    model.eval()
    fl, ll = [], []
    for x, y in loader:
        fl.append(model.get_features(x.to(DEVICE, non_blocking=True)))
        ll.append(y.to(DEVICE, non_blocking=True))
    H = torch.cat(fl); Y = torch.cat(ll)
    mu_G = H.mean(0)
    mu_c = torch.stack([H[Y==c].mean(0) for c in range(K)])
    M    = mu_c - mu_G
    Sw   = sum((H[Y==c]-mu_c[c]).T@(H[Y==c]-mu_c[c]) for c in range(K))/len(H)
    Sb   = M.T @ M / K
    nc1  = (torch.trace(Sw)/torch.trace(Sb).clamp(1e-10)).item()
    Mn   = F.normalize(M, dim=1)
    cos  = Mn @ Mn.T
    mask = ~torch.eye(K, dtype=torch.bool, device=DEVICE)
    nc2  = (cos[mask]-(-1./(K-1))).abs().mean().item()
    Wn   = F.normalize(model.get_classifier_weights().to(DEVICE), dim=1)
    nc3  = (1-(Mn*Wn).sum(1).mean()).item()
    return {'nc1':nc1,'nc2':nc2,'nc3':nc3,'feat_norm':H.norm(dim=1).mean().item()}

def evaluate(model, loader):
    model.eval(); correct=total=0
    with torch.no_grad():
        for x, y in loader:
            x,y = x.to(DEVICE,non_blocking=True), y.to(DEVICE,non_blocking=True)
            correct += (model(x).argmax(1)==y).sum().item()
            total   += len(y)
    return correct/total

print('Metrics ready.')


Metrics ready.


In [5]:
def run(model, name, lr=1e-3, wd=1e-4,
        phase1=200, phase2=600, nc_every=10, nc_thresh=0.01):
    try:
        model = torch.compile(model, mode='reduce-overhead')
    except Exception:
        pass
    model = model.to(DEVICE)
    K=10; rows=[]; terminal=False; t0=time.time()
    for phase, loss_fn, n_ep in [(1,'ce',phase1),(2,'mse',phase2)]:
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_ep)
        off = phase1 if phase==2 else 0
        for ep_l in range(1, n_ep+1):
            ep = off + ep_l
            model.train()
            for x, y in train_loader:
                x,y = x.to(DEVICE,non_blocking=True), y.to(DEVICE,non_blocking=True)
                opt.zero_grad(set_to_none=True)
                logits = model(x)
                loss = (F.mse_loss(logits, F.one_hot(y,K).float())
                        if loss_fn=='mse' else F.cross_entropy(logits, y))
                loss.backward(); opt.step()
            sch.step()
            if ep_l % nc_every == 0 or ep_l == n_ep:
                tr = evaluate(model, train_loader)
                te = evaluate(model, test_loader)
                if tr >= 0.99 and not terminal:
                    terminal = True
                    print(f'  [{name}] Terminal ep={ep}  test={te:.4f}')
                nc = (compute_nc(model, train_loader) if terminal
                      else {'nc1':None,'nc2':None,'nc3':None,'feat_norm':None})
                rows.append({'epoch':ep,'phase':phase,'train':tr,'test':te,**nc})
                nc1s = f"{nc['nc1']:.5f}" if nc['nc1'] is not None else 'N/A'
                fns  = f"{nc['feat_norm']:.3f}" if nc['feat_norm'] else 'N/A'
                print(f'  ep={ep:>4} tr={tr:.4f} nc1={nc1s} fn={fns} '
                      f't={(time.time()-t0)/60:.1f}m')
                if nc['nc1'] is not None and nc['nc1'] < nc_thresh:
                    print(f'  *** T_NC={ep}  fn={nc["feat_norm"]:.4f}')
                    return pd.DataFrame(rows), ep, nc['feat_norm']
    return pd.DataFrame(rows), None, None

print('run() ready.  Phase-2 budget: 600 epochs.')


run() ready.  Phase-2 budget: 600 epochs.


In [6]:
results = []
for seed in range(3):
    print(f'\n=== MLP-5 CIFAR-10 seed={seed} ===')
    torch.manual_seed(seed)
    model = MLP(in_dim=3*32*32, width=512)
    df, t_nc, fn = run(model, f'mlp5-cifar-s{seed}')
    df.to_csv(f'/tmp/mlp5_cifar10_s{seed}.csv', index=False)
    fn_val = float(fn) if fn is not None else None
    nc_rows = df.dropna(subset=['feat_norm'])
    fn_final = nc_rows.feat_norm.iloc[-1] if len(nc_rows) else None
    results.append({'seed':seed,'T_NC':t_nc,'fn':fn_val,'fn_final':fn_final,
                    'test_acc':df.test.iloc[-1]})
    status = f'T_NC={t_nc}  fn={fn_val:.4f}' if t_nc else f'DNF  fn_final={fn_final:.4f}'
    print(f'  => {status}')

df_res = pd.DataFrame(results)
df_res.to_csv('/tmp/mlp5_cifar10_summary.csv', index=False)
print('\n=== RESULTS ===')
print(df_res.to_string())

confirmed = df_res.dropna(subset=['fn'])
if len(confirmed) > 0:
    mean_fn = confirmed.fn.mean()
    std_fn  = confirmed.fn.std() if len(confirmed)>1 else 0.0
    print(f'\nMLP-5 CIFAR-10:  fn={mean_fn:.4f} +/- {std_fn:.4f}  N={len(confirmed)}')
    print(f'\nThreshold comparison:')
    print(f'  MNIST  MLP-5    (arch=MLP,  data=MNIST):  fn={MNIST_MLP5_FN:.3f}')
    print(f'  CIFAR  MLP-5    (arch=MLP,  data=CIFAR):  fn={mean_fn:.3f}  <-- this')
    print(f'  CIFAR  ResNet20 (arch=ResN, data=CIFAR):  fn={CIFAR_RESNET_FN:.3f}')
    gap_arch = abs(mean_fn - MNIST_MLP5_FN)   # same arch, diff data
    gap_data = abs(mean_fn - CIFAR_RESNET_FN)  # same data, diff arch
    print(f'\n  CIFAR MLP-5 vs MNIST MLP-5 (data effect):  {gap_arch:.3f}')
    print(f'  CIFAR MLP-5 vs CIFAR ResNet (arch effect): {gap_data:.3f}')
    if gap_arch > gap_data:
        print('  → Dataset effect > Architecture effect')
    elif gap_data > gap_arch:
        print('  → Architecture effect > Dataset effect')
    else:
        print('  → Both effects roughly equal')



=== MLP-5 CIFAR-10 seed=0 ===
  ep=  10 tr=0.7832 nc1=N/A fn=N/A t=0.7m
  ep=  20 tr=0.9264 nc1=N/A fn=N/A t=1.3m
  ep=  30 tr=0.9642 nc1=N/A fn=N/A t=1.9m
  ep=  40 tr=0.9670 nc1=N/A fn=N/A t=2.5m
  ep=  50 tr=0.9837 nc1=N/A fn=N/A t=3.1m
  ep=  60 tr=0.9748 nc1=N/A fn=N/A t=3.6m
  ep=  70 tr=0.9796 nc1=N/A fn=N/A t=4.2m
  ep=  80 tr=0.9883 nc1=N/A fn=N/A t=4.8m
  ep=  90 tr=0.9864 nc1=N/A fn=N/A t=5.4m
  [mlp5-cifar-s0] Terminal ep=100  test=0.5378
  ep= 100 tr=0.9980 nc1=1.76598 fn=37.512 t=6.0m
  ep= 110 tr=0.9995 nc1=1.59563 fn=38.149 t=6.6m
  ep= 120 tr=1.0000 nc1=1.60773 fn=42.611 t=7.3m
  ep= 130 tr=0.9997 nc1=1.46780 fn=30.893 t=7.9m
  ep= 140 tr=1.0000 nc1=1.51161 fn=41.681 t=8.5m
  ep= 150 tr=1.0000 nc1=1.55593 fn=42.222 t=9.1m
  ep= 160 tr=1.0000 nc1=1.36465 fn=34.677 t=9.8m
  ep= 170 tr=1.0000 nc1=1.41433 fn=38.996 t=10.4m
  ep= 180 tr=1.0000 nc1=1.41673 fn=39.139 t=11.0m
  ep= 190 tr=1.0000 nc1=1.41061 fn=38.959 t=11.7m
  ep= 200 tr=1.0000 nc1=1.40847 fn=38.889 t=12.3m
 

In [7]:
import os
for f in ['/tmp/mlp5_cifar10_summary.csv'] + \
         [f'/tmp/mlp5_cifar10_s{s}.csv' for s in range(3)]:
    if os.path.exists(f): files.download(f)
print('Done.')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done.
